## **eggNOG-mapper**

Будем использовать аднынй тул для расширенной функциональной аннотации:

    К какой группе эволюционно родственных белков он принадлежит?

    Какие функции известны у его ортологов?

    Участвует ли белок в определённых биологических процессах?

    Является ли он ферментом?

    какими метаболическими путями он связан?

In [ ]:
from pathlib import Path
import pandas as pd
from io import StringIO

project_dir = (Path.home()/ "Desktop/bioinformatics/projects/genome_annotation")

file_path = (project_dir/ "results/functional/eggnog/query.emapper.annotations")

# Удаляем служебные строчки 
# А заголовок таблицы обычно начинается с #query
with open(file_path) as f:
    lines = [line for line in f if not line.startswith("##")]

eggnog_df = pd.read_csv(
    StringIO("".join(lines)),
    sep="\t",
    na_values=["-"]
)
eggnog_df.columns = eggnog_df.columns.str.lstrip("#")

print("Количество строк:", len(eggnog_df))
print("Количество столбцов:", len(eggnog_df.columns))
print("\nНазвания столбцов:")
print(eggnog_df.columns.tolist())

display(eggnog_df.head())

Количество строк: 4224
Количество столбцов: 22

Названия столбцов:
['query', 'seed_ortholog', 'evalue', 'score', 'eggNOG_OGs', 'tax_ceiling', 'farthest_donor_lineage', 'COG_category', 'Preferred_name', 'GOs', 'EC', 'KEGG_ko', 'KEGG_Pathway', 'KEGG_Module', 'KEGG_Reaction', 'KEGG_rclass', 'BRITE', 'KEGG_TC', 'CAZy', 'BiGG_Reaction', 'PFAMs', 'annotation_confidence']


,query,seed_ortholog,evalue,score,eggNOG_OGs,tax_ceiling,farthest_donor_lineage,COG_category,Preferred_name,GOs,...,KEGG_Pathway,KEGG_Module,KEGG_Reaction,KEGG_rclass,BRITE,KEGG_TC,CAZy,BiGG_Reaction,PFAMs,annotation_confidence
0,KOIAIB_01159,357240.GCA_001063355_00271,1.910000e-44,145.0,PP-binding@131567|cA-12,Prokaryota,1;131567;2;2323;1783234;1817807;3106474;1817870,COG0236,acpP,"GO:0000035,GO:0000036,GO:0005515,GO:0005737,GO...",...,"01100,01110","M00848,M00885,M00886",NaN,NaN,00001,NaN,NaN,NaN,PP-binding_6_73,mm-hhh--h---h
1,KOIAIB_03833,357240.GCA_001063355_01138,6.810000e-53,167.0,RNA_pol_Rpb6@131567|A-1,Prokaryota,1;131567;2;3379134;1783257;203682;473814;1801951,COG1758,rpoZ,"GO:0000345,GO:0001000,GO:0005829,GO:0006352,GO...",...,03020,NaN,NaN,NaN,"00001,01000,01611,03021,03400",NaN,NaN,NaN,RNA_pol_Rpb6_9_59,mmhhh---h---h
2,KOIAIB_03480,357240.GCA_001063355_01353,5.050000e-162,455.0,"KH_2@131567|O-5,Ribosomal_S3_C@131567|A-1",Prokaryota,1;131567;2;3379134;1783257;203682;203683;112;4...,COG0092,rpsC,"GO:0000028,GO:0002181,GO:0003735,GO:0005737,GO...",...,03010,NaN,NaN,NaN,"00001,01610,03011",NaN,NaN,NaN,"KH_2_38_111,Ribosomal_S3_C_120_202",mm-hh---h---h
3,KOIAIB_03477,357240.GCA_001063355_01356,1.230000e-54,171.0,Ribosomal_S17@131567|A-1,Prokaryota,1;131567;2;3379134;1783257;204428;204429;51291...,COG0186,rpsQ,"GO:0000028,GO:0002181,GO:0003735,GO:0005737,GO...",...,03010,NaN,NaN,NaN,"00001,01610,03011",NaN,NaN,NaN,Ribosomal_S17_10_76,mh-hh---h---h
4,KOIAIB_03927,357240.GCA_001063355_01707,8.350000e-44,144.0,ATP-synt_C@131567|A-1,Prokaryota,1;131567;2;1783272;201174;1760;85007;1762;6705...,COG0636,atpE,"GO:0005886,GO:0042777,GO:0045259,GO:0046933,GO...",...,"00190,01100",M00157,NaN,NaN,"00001,00194",NaN,NaN,NaN,ATP-synt_C_11_71,mm-hhh--h---h


In [3]:
from Bio import SeqIO

bakta_proteins = list(SeqIO.parse(project_dir /"results/structural/bakta/ecoli.faa", "fasta"))

bakta_ids = {p.id for p in bakta_proteins}
eggnog_ids = set(eggnog_df["query"].dropna().astype(str))

print("Белков Bakta:", len(bakta_ids))
print("Белков в таблице eggNOG:", len(eggnog_ids))
print("Совпадающих ID:", len(bakta_ids & eggnog_ids))
print("Нет в eggNOG:", len(bakta_ids - eggnog_ids))

Белков Bakta: 4315
Белков в таблице eggNOG: 4224
Совпадающих ID: 4224
Нет в eggNOG: 91


In [7]:
#Объединение аннотации Bakta и eggNOG
project_dir = Path.home() / ("Desktop/bioinformatics/projects/genome_annotation")

bakta_dir = project_dir / "results/structural/bakta"

genome = SeqIO.read(
    bakta_dir / "ecoli.gbff",
    "genbank"
)

bakta_data = []

for feature in genome.features:
    if feature.type != "CDS":
        continue

    bakta_data.append({
        "query": feature.qualifiers.get("locus_tag", [None])[0],
        "bakta_gene": feature.qualifiers.get("gene", [None])[0],
        "bakta_product": feature.qualifiers.get("product", [None])[0],
        "start": int(feature.location.start) + 1,
        "end": int(feature.location.end),
        "strand": feature.location.strand
    })

bakta_table = pd.DataFrame(bakta_data)

print("Совпадающих ID:", len(set(bakta_table["query"]) & eggnog_ids))

merged = bakta_table.merge(
    eggnog_df,
    on = "query",
    how = 'left',
    validate = 'one_to_one'
)

print("Всего CDS:", len(merged))

display(merged.head())

Совпадающих ID: 4224
Всего CDS: 4315


,query,bakta_gene,bakta_product,start,end,strand,seed_ortholog,evalue,score,eggNOG_OGs,...,KEGG_Pathway,KEGG_Module,KEGG_Reaction,KEGG_rclass,BRITE,KEGG_TC,CAZy,BiGG_Reaction,PFAMs,annotation_confidence
0,KOIAIB_00001,NaN,hypothetical protein,3,98,1,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,KOIAIB_00002,NaN,thr operon leader peptide,190,255,1,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,KOIAIB_00003,NaN,Bifunctional aspartokinase/homoserine dehydrog...,337,2799,1,585057.ECIAI39_0001,0.000000e+00,1589.0,"Homoserine_dh@131567|A-1,NAD_binding_3@131567|...",...,"00260,00261,00270,00300,01100,01110,01120,01230","M00016,M00017,M00018,M00526,M00527",NaN,NaN,"00001,01000",NaN,NaN,NaN,"AA_kinase_2_284,ACT_7_314_375,ACT_7_393_453,Ho...",hhhhhh--h---h
3,KOIAIB_00004,NaN,Homoserine kinase,2801,3733,1,511145.b0003,3.280000e-226,624.0,"GHMP_kinases_N@131567|Ap-7,GHMP_kinases_C@1315...",...,"00260,00270,01100,01110,01120,01230",M00018,NaN,NaN,"00001,01000",NaN,NaN,NaN,"GHMP_kinases_C_212_281,GHMP_kinases_N_85_150",mhhhhh--h---h
4,KOIAIB_00005,NaN,Threonine synthase,3734,5020,1,511145.b0004,5.450000e-312,850.0,"PALP@131567|HD-13!,Thr_synth_N@131567|A-1",...,"00260,00750,01100,01110,01120,01230",M00018,NaN,NaN,"00001,01000",NaN,NaN,NaN,"PALP_99_369,Thr_synth_N_8_80",hhhhhh--h---h


In [11]:
hypothetical = merged[merged["bakta_product"].fillna("").str.lower() == "hypothetical protein"].copy()

print("Hypothetical proteins Bakta:", len(hypothetical))

for column in ["Description", "GOs", "KEGG_ko", "PFAMs"]:
    if column in hypothetical.columns:
        print(
            column,
            hypothetical[column].notna().sum()
        )

Hypothetical proteins Bakta: 234
GOs 73
KEGG_ko 96
PFAMs 136


        Pfam - белковые семейтсва и домены. Указывают на домен неизвестной функции (DUF); 

        KEGG KO - функциональные ортологические группы. Оцениват функцию по ортологам;

        GO - функциональные термины. Что делает ('связывает ионы, условно')

In [9]:
#Сохранение
output = (project_dir/ "results/integrated/ecoli_functional_annotation.tsv")

output.parent.mkdir(parents=True, exist_ok=True)

merged.to_csv(output, sep="\t", index=False)

print("Сохранено:", output)

Сохранено: /Users/danasokol/Desktop/bioinformatics/projects/genome_annotation/results/integrated/ecoli_functional_annotation.tsv


In [12]:
# Гипотетические белки с дополнительными аннотациями
columns = [
    "query",
    "bakta_gene",
    "bakta_product",
    "Description",
    "GOs",
    "KEGG_ko",
    "PFAMs"
]

available_columns = [ c for c in columns if c in hypothetical.columns]

annotated_hypothetical = hypothetical[
    hypothetical[
        [c for c in ["GOs", "KEGG_ko", "PFAMs"]
         if c in hypothetical.columns]
    ].notna().any(axis=1)
].copy()

print(
    "Hypothetical proteins с хотя бы одной аннотацией:",
    len(annotated_hypothetical)
)

display(annotated_hypothetical[available_columns].head(15))

Hypothetical proteins с хотя бы одной аннотацией: 153


,query,bakta_gene,bakta_product,GOs,KEGG_ko,PFAMs
5,KOIAIB_00006,NaN,hypothetical protein,NaN,NaN,DUF2502_25_73
12,KOIAIB_00013,NaN,hypothetical protein,NaN,NaN,DUF2541_3_132
32,KOIAIB_00036,NaN,hypothetical protein,NaN,K08277,NaN
68,KOIAIB_00075,NaN,hypothetical protein,"GO:0004857,GO:0046325",NaN,NaN
112,KOIAIB_00124,NaN,hypothetical protein,NaN,NaN,DUF3300_75_351
185,KOIAIB_00199,NaN,hypothetical protein,NaN,K19000,ROF_9_83
195,KOIAIB_00210,NaN,hypothetical protein,"GO:0005515,GO:0005886,GO:0015191,GO:0015821,GO...",K02072,BPD_transp_1_32_216
201,KOIAIB_00223,NaN,hypothetical protein,NaN,NaN,Methyltransf_11_6_96
218,KOIAIB_00242,NaN,hypothetical protein,"GO:0003674,GO:0008150",K19303,NLPC_P60_131_241
221,KOIAIB_00247,NaN,hypothetical protein,NaN,K02557,OmpA_70_163


In [13]:
output = (
    project_dir
    / "results/functional/eggnog/hypothetical_annotated.tsv"
)

annotated_hypothetical[available_columns].to_csv(
    output,
    sep="\t",
    index=False
)